In [ ]:
import numpy as np
import pandas as pd

In [31]:
shap_concat = np.load('/data/code/mywork/gas_sensor/weights/RGB_diff_refine_label_new_n2/shap_values.npy')
X_concat = np.load('/data/code/mywork/gas_sensor/weights/RGB_diff_refine_label_new_n2/shap_X_val.npy')
shap_concat.shape, X_concat.shape
feature_names = [f'Dye{i}' for i in range(1, 9)]

In [32]:
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_concat).mean(axis=0),
})
feature_importance['rank'] = feature_importance['mean_abs_shap'].rank(ascending=False, method='min').astype(int)
feature_importance = feature_importance.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
feature_importance = feature_importance[['rank', 'feature', 'mean_abs_shap']]

print(feature_importance.to_string(index=False))

 rank feature  mean_abs_shap
    1    Dye6      11.932959
    2    Dye8       4.579290
    3    Dye1       4.200818
    4    Dye3       3.899022
    5    Dye7       3.892088
    6    Dye2       3.066661
    7    Dye5       3.012001
    8    Dye4       0.735347


In [33]:
data_path = '/data/jxiao/gas_data/gas_new/values_refine/RGB_diff_refine_label_new_n2.csv'
diff = pd.read_csv(data_path, index_col='id')
diff.columns = [f'Dye{i}' for i in range(1, 9)] + ['label']
X = diff.drop('label', axis=1, inplace=False).to_numpy()
y = diff['label'].to_numpy().astype(int)
print(len(y), sum(y))

134 81


In [36]:
import matplotlib.pyplot as plt
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, roc_curve, precision_recall_curve
import os

INPUT_DIM = 8
EPOCHS = 250
BATCH_SIZE = 20
LR = 2e-4
PATIENCE = 10  # early stopping
N_SPLITS = 5
N_PERMUTATION_REPEATS = 30
PERMUTATION_RANDOM_STATE = 42
THRESHOLD = 0.5
DROPOUT = 0.3
weight_dir = ''
use_smote = False
use_posweight = False
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if weight_dir:
    if not os.path.exists(weight_dir):
        os.makedirs(weight_dir)
        
class SmallMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),  # 64
            nn.ReLU(),
            nn.Dropout(DROPOUT),  # 0.3
            nn.Linear(64, 32),  # 32
            nn.ReLU(),
            nn.Dropout(DROPOUT),  # 0.3
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

def evaluate_mlp(model, X_eval, y_eval_np):
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(X_eval.to(device))).cpu().numpy().ravel()
    pred_label = (prob > THRESHOLD).astype(int)
    metrics = {
        'acc': accuracy_score(y_eval_np, pred_label),
        'f1': f1_score(y_eval_np, pred_label, zero_division=0),
        'auc': roc_auc_score(y_eval_np, prob),
        'precision': precision_score(y_eval_np, pred_label, zero_division=0),
        'recall': recall_score(y_eval_np, pred_label, zero_division=0),
    }
    return metrics, prob, pred_label

In [37]:
skf = StratifiedKFold(n_splits=N_SPLITS, 
                      shuffle=True, 
                      random_state=42
                    )

all_acc, all_f1, all_auc, all_precision, all_recall = [], [], [], [], [] # eval metrics
all_fpr, all_tpr = [], []  # ROC curves
all_precision_thre, all_recall_thre, all_thresholds = [], [], []  # Precision / Recall vs. Threshold curves
y_val_list, y_pred_list, y_label_list, prob_list = [], [], [], []  # save ground truth and predictions
permutation_rows = []
acc_per_epoch, f1_per_epoch, auc_per_epoch = [[] for _ in range(N_SPLITS)], [[] for _ in range(N_SPLITS)], [[] for _ in range(N_SPLITS)] # model performance evolution curves
train_loss_list, val_loss_list= [[] for _ in range(N_SPLITS)], [[] for _ in range(N_SPLITS)] # loss curves

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"\n===== Fold {fold + 1} =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # --- normalization ---
    scaler = StandardScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    y_val_np = y_val.squeeze().cpu().numpy().astype(int)

    # --- DataLoader ---
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

    # --- Model & Optimizer ---
    model = SmallMLP(INPUT_DIM).to(device)
    if use_posweight:
        num_pos = y_train.sum()
        num_neg = y_train.shape[0] - num_pos
        # pos_weight = torch.tensor([num_neg / num_pos]).to(device)  # 正例的权重
        pos_weight = torch.tensor([3.0]).to(device)  # 手动设定正例权重
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    # optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)

    # --- Train ---
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)

        #--- Validation ---
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
                all_preds.append(torch.sigmoid(preds).cpu().numpy())
                all_labels.append(yb.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        pred_labels = (all_preds > 0.5).astype(int)

        acc = accuracy_score(all_labels, pred_labels)
        f1 = f1_score(all_labels, pred_labels)
        auc = roc_auc_score(all_labels, all_preds)

        acc_per_epoch[fold].append(acc)
        f1_per_epoch[fold].append(f1)
        auc_per_epoch[fold].append(auc)

        train_loss_list[fold].append(train_loss)
        val_loss_list[fold].append(val_loss)

        #--- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE or epoch == EPOCHS - 1:
                print("Stop in epoch {}!".format(epoch + 1))
    model.load_state_dict(best_weights)

    # --- Final evaluation ---
    baseline_metrics, preds, pred_labels = evaluate_mlp(model, X_val, y_val_np)
    prob_list.append(preds)

    acc = baseline_metrics['acc']
    f1 = baseline_metrics['f1']
    auc = baseline_metrics['auc']
    precision = baseline_metrics['precision']
    recall = baseline_metrics['recall']
    
    fpr, tpr, _ = roc_curve(y_val_np, preds)
    precision_thre, recall_thre, thresholds = precision_recall_curve(y_val_np, preds)

    X_val_np = X_val.cpu().numpy()
    rng = np.random.default_rng(PERMUTATION_RANDOM_STATE + fold)
    for feature_idx, feature in enumerate(diff.columns[:-1].tolist()):
        repeat_metrics = []
        for _ in range(N_PERMUTATION_REPEATS):
            X_perm = X_val_np.copy()
            X_perm[:, feature_idx] = rng.permutation(X_perm[:, feature_idx])
            X_perm = torch.tensor(X_perm, dtype=torch.float32)
            perm_metrics, _, _ = evaluate_mlp(model, X_perm, y_val_np)
            repeat_metrics.append(perm_metrics)
        for metric_name in baseline_metrics:
            perm_values = np.array([m[metric_name] for m in repeat_metrics])
            permutation_rows.append({
                'fold': fold + 1,
                'feature': feature,
                'metric': metric_name,
                'baseline': baseline_metrics[metric_name],
                'permuted_mean': perm_values.mean(),
                'permuted_std': perm_values.std(),
                'importance': baseline_metrics[metric_name] - perm_values.mean(),
            })

    print(f"Fold {fold + 1} val results: ACC={acc:.3f}, F1={f1:.3f}, precision={precision:.3f}, recall={recall:.3f}")
    all_acc.append(acc)
    all_f1.append(f1)
    all_auc.append(auc)
    all_precision.append(precision)
    all_recall.append(recall)

    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_precision_thre.append(precision_thre)
    all_recall_thre.append(recall_thre)
    all_thresholds.append(thresholds)

    y_val_list.append(y_val.squeeze().cpu().tolist())
    y_pred_list.append(preds.squeeze().tolist())
    y_label_list.append(pred_labels.squeeze().tolist())


===== Fold 1 =====
Stop in epoch 174!
Stop in epoch 175!
Stop in epoch 176!
Stop in epoch 177!
Stop in epoch 178!
Stop in epoch 179!
Stop in epoch 180!
Stop in epoch 181!
Stop in epoch 215!
Stop in epoch 216!
Stop in epoch 217!
Stop in epoch 218!
Stop in epoch 219!
Stop in epoch 220!
Stop in epoch 221!
Stop in epoch 222!
Stop in epoch 223!
Stop in epoch 224!
Stop in epoch 225!
Stop in epoch 226!
Stop in epoch 227!
Stop in epoch 228!
Stop in epoch 229!
Stop in epoch 230!
Stop in epoch 231!
Stop in epoch 232!
Stop in epoch 233!
Stop in epoch 234!
Stop in epoch 235!
Stop in epoch 236!
Stop in epoch 237!
Stop in epoch 238!
Stop in epoch 239!
Stop in epoch 240!
Stop in epoch 241!
Stop in epoch 242!
Stop in epoch 243!
Stop in epoch 244!
Stop in epoch 245!
Stop in epoch 246!
Stop in epoch 247!
Stop in epoch 248!
Stop in epoch 249!
Stop in epoch 250!
Fold 1 val results: ACC=0.889, F1=0.914, precision=0.889, recall=0.941

===== Fold 2 =====
Stop in epoch 107!
Stop in epoch 108!
Stop in epoch 1

In [40]:
permutation_detail = pd.DataFrame(permutation_rows)
permutation_importance = (
    permutation_detail
    .groupby(['feature', 'metric'], as_index=False)
    .agg(
        baseline_mean=('baseline', 'mean'),
        permuted_mean=('permuted_mean', 'mean'),
        importance_mean=('importance', 'mean'),
        importance_std=('importance', 'std'),
    )
)
permutation_importance['rank'] = permutation_importance.groupby('metric')['importance_mean'].rank(ascending=False, method='min').astype(int)
permutation_importance = permutation_importance.sort_values(['metric', 'rank']).reset_index(drop=True)

permutation_auc_importance = (
    permutation_importance[permutation_importance['metric'] == 'auc']
    .sort_values('importance_mean', ascending=False)
    .reset_index(drop=True)
)

output_dir = '/data/code/mywork/gas_sensor/weights/RGB_diff_refine_label_new_n2'
os.makedirs(output_dir, exist_ok=True)
permutation_detail.to_csv(os.path.join(output_dir, 'mlp_permutation_importance_detail.csv'), index=False)
permutation_importance.to_csv(os.path.join(output_dir, 'mlp_permutation_importance.csv'), index=False)

print('===== MLP Permutation Importance Ranking (AUC drop) =====')
print(permutation_auc_importance[['rank', 'feature', 'importance_mean', 'importance_std', 'baseline_mean', 'permuted_mean']].to_string(index=False))

===== MLP Permutation Importance Ranking (AUC drop) =====
 rank feature  importance_mean  importance_std  baseline_mean  permuted_mean
    1    Dye6         0.055257        0.042666       0.861143       0.805886
    2    Dye1         0.051437        0.024019       0.861143       0.809706
    3    Dye3         0.050109        0.030791       0.861143       0.811034
    4    Dye5         0.036621        0.029290       0.861143       0.824522
    5    Dye8         0.036495        0.025827       0.861143       0.824648
    6    Dye2         0.019086        0.016089       0.861143       0.842057
    7    Dye7         0.011360        0.005324       0.861143       0.849783
    8    Dye4        -0.002034        0.012070       0.861143       0.863177


In [41]:
print(feature_importance.to_string(index=False))

 rank feature  mean_abs_shap
    1    Dye6      11.932959
    2    Dye8       4.579290
    3    Dye1       4.200818
    4    Dye3       3.899022
    5    Dye7       3.892088
    6    Dye2       3.066661
    7    Dye5       3.012001
    8    Dye4       0.735347
